# Floquet edge--TLS channel and position validation (targeted-Arnoldi v3)

This notebook tests the two remaining links in the theory chain:

\[
B_{0\pi}^{(m)}(j)
\longrightarrow g_m(j)
\longrightarrow \lambda_\pi
\longrightarrow \text{observable position/frequency response}.
\]

It preserves the model and working point used by the completed production scans. Cells expected to exceed five minutes are disabled and checkpointed for local execution. A fixed-frequency TLS amplitude is **not** identified with a localization length; the dressed matrix element and the full frequency response are tested separately.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-floquet-tls")

from dataclasses import dataclass, replace
from pathlib import Path
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import eig, expm, schur
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.sparse import csr_matrix, eye, kron
from scipy.sparse.linalg import LinearOperator, eigs, expm_multiply

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

RUN_THRESHOLD_COMPARISON = True
RUN_DRESSED_MATRIX_ELEMENT = True
RUN_REDUCED_AND_N3_CHANNEL = True
RUN_N4_POSITION_PILOT = True

RUN_EXACT_N4_CHANNEL = True
RUN_N4_ARNOLDI_VALIDATION = True
RUN_LONG_N6_POSITION_GRID = False
RUN_LONG_N6_MATRIX_FREE_CHANNEL = False

CELL_TIMEOUT_POLICY_SECONDS = 300

# Put long-run checkpoints in one explicit directory. For persistent Colab runs,
# change this to a mounted Drive directory. Local users may use, for example,
# Path(r"D:\Floquet_TLS\checkpoints").
CHECKPOINT_DIRECTORY = Path(
    os.environ.get("FLOQUET_TLS_CHECKPOINT_DIR", ".")
).expanduser().resolve()

print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "cell_timeout_policy_s": CELL_TIMEOUT_POLICY_SECONDS,
    "long_N6_position_grid": RUN_LONG_N6_POSITION_GRID,
    "long_N6_matrix_free_channel": RUN_LONG_N6_MATRIX_FREE_CHANNEL,
    "exact_N4_channel": RUN_EXACT_N4_CHANNEL,
    "N4_Arnoldi_validation": RUN_N4_ARNOLDI_VALIDATION,
    "checkpoint_directory": str(CHECKPOINT_DIRECTORY),
})


## 1. Unified model with a movable TLS contact

For a TLS coupled to chain site (j_d),

\[
H_{ed}(j_d)=g\left(s_+^{j_d}\tau_-+s_-^{j_d}\tau_+\right)
=\frac g2\left(X_{j_d}X_d+Y_{j_d}Y_d\right).
\]

The drive, TLS Hamiltonian, and TLS-only dissipator are unchanged. The default contact is (j_d=0). Moving this microscopic spin coupling into the bulk is a distinct operation from directly coupling a fermion to a Majorana wavefunction; therefore its spatial response must be calculated rather than assumed.


In [ ]:
I2 = csr_matrix(np.eye(2, dtype=complex))
X2 = csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex))
Y2 = csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=complex))
Z2 = csr_matrix(np.diag([1.0, -1.0]).astype(complex))
SM2 = csr_matrix(np.array([[0, 1], [0, 0]], dtype=complex))
SP2 = SM2.getH()


def kron_all(factors):
    output = csr_matrix([[1.0 + 0.0j]])
    for factor in factors:
        output = kron(output, factor, format="csr")
    return output


def site_operator(local_operator, site, n_total):
    return kron_all([
        local_operator if index == site else I2
        for index in range(n_total)
    ])


def operator_lists(n_total):
    return {
        "x": [site_operator(X2, site, n_total) for site in range(n_total)],
        "y": [site_operator(Y2, site, n_total) for site in range(n_total)],
        "z": [site_operator(Z2, site, n_total) for site in range(n_total)],
        "sm": [site_operator(SM2, site, n_total) for site in range(n_total)],
    }


def zero_operator(dimension):
    return csr_matrix((dimension, dimension), dtype=complex)


def computational_ket(bits):
    index = 0
    for bit in bits:
        index = 2 * index + int(bit)
    ket = np.zeros(2 ** len(bits), dtype=complex)
    ket[index] = 1.0
    return ket


def density_vector_from_bits(bits):
    ket = computational_ket(bits)
    return np.outer(ket, ket.conjugate()).reshape(-1, order="F")


def unvec(vector, dimension):
    return np.asarray(vector).reshape((dimension, dimension), order="F")


def expectation_from_vec(operator, vector, dimension):
    rho = unvec(vector, dimension)
    return float(np.trace(operator.toarray() @ rho).real)


def liouvillian(hamiltonian, collapse_operators):
    dimension = hamiltonian.shape[0]
    identity = eye(dimension, format="csr", dtype=complex)
    generator = -1j * (
        kron(identity, hamiltonian, format="csr")
        - kron(hamiltonian.T, identity, format="csr")
    )
    for collapse in collapse_operators:
        cdc = collapse.getH() @ collapse
        generator = generator + kron(collapse.conjugate(), collapse, format="csr")
        generator = generator - 0.5 * kron(identity, cdc, format="csr")
        generator = generator - 0.5 * kron(cdc.T, identity, format="csr")
    return generator.tocsr()


@dataclass(frozen=True)
class Parameters:
    N: int = 4
    J: float = 1.0
    h: float = 1.0
    alpha_over_pi: float = 0.75
    beta_over_pi: float = 0.90
    g: float = 0.08
    omega_d: float | None = None
    gamma1: float = 0.08
    gamma_phi: float = 0.0
    periods: int = 60
    samples_per_step: int = 2
    tls_site: int = 0

    @property
    def alpha(self):
        return self.alpha_over_pi * np.pi

    @property
    def beta(self):
        return self.beta_over_pi * np.pi

    @property
    def T1(self):
        return self.beta / (2.0 * self.J)

    @property
    def T2(self):
        return self.alpha / (2.0 * self.h)

    @property
    def T(self):
        return self.T1 + self.T2

    @property
    def Omega(self):
        return 2.0 * np.pi / self.T

    @property
    def tls_frequency(self):
        return self.Omega / 2.0 if self.omega_d is None else self.omega_d


p = Parameters()
print(p)
print({"T": p.T, "Omega": p.Omega, "Omega_over_2": p.Omega / 2.0})


In [ ]:
_STATIC_MODEL_CACHE = {}


def static_chain_data(N, J, h):
    key = (int(N), float(J), float(h))
    if key not in _STATIC_MODEL_CACHE:
        n_total = N + 1
        ops = operator_lists(n_total)
        dimension = 2 ** n_total
        h_zz = zero_operator(dimension)
        h_x = zero_operator(dimension)
        for site in range(N - 1):
            h_zz = h_zz - J * (ops["z"][site] @ ops["z"][site + 1])
        for site in range(N):
            h_x = h_x - h * ops["x"][site]
        _STATIC_MODEL_CACHE[key] = (ops, h_zz.tocsr(), h_x.tocsr(), dimension)
    return _STATIC_MODEL_CACHE[key]


def build_open_model(parameters):
    if not 0 <= parameters.tls_site < parameters.N:
        raise ValueError("tls_site must be a chain-site index")
    d_site = parameters.N
    ops, h_zz, h_x, dimension = static_chain_data(
        parameters.N, parameters.J, parameters.h
    )
    h_d = -0.5 * parameters.tls_frequency * ops["z"][d_site]
    h_ed = parameters.g * (
        ops["sm"][parameters.tls_site].getH() @ ops["sm"][d_site]
        + ops["sm"][parameters.tls_site] @ ops["sm"][d_site].getH()
    )
    h_xy = 0.5 * parameters.g * (
        ops["x"][parameters.tls_site] @ ops["x"][d_site]
        + ops["y"][parameters.tls_site] @ ops["y"][d_site]
    )
    collapse_operators = []
    if parameters.gamma1 > 0:
        collapse_operators.append(np.sqrt(parameters.gamma1) * ops["sm"][d_site])
    if parameters.gamma_phi > 0:
        collapse_operators.append(
            np.sqrt(parameters.gamma_phi / 2.0) * ops["z"][d_site]
        )
    h1 = (h_zz + h_d + h_ed).tocsr()
    h2 = (h_x + h_d + h_ed).tocsr()
    identity = eye(dimension, format="csr", dtype=complex)
    tau_z = -ops["z"][d_site]
    return {
        "dimension": dimension,
        "ops": ops,
        "H1": h1,
        "H2": h2,
        "collapse": collapse_operators,
        "observables": {
            "edge_left": ops["z"][0],
            "edge_right": ops["z"][parameters.N - 1],
            "tls_x": ops["x"][d_site],
            "tls_y": ops["y"][d_site],
            "tls_excited": (0.5 * (identity + tau_z)).tocsr(),
        },
        "exchange_identity_error": np.linalg.norm((h_ed - h_xy).toarray()),
    }


def simulate_open(parameters, validate=False):
    model = build_open_model(parameters)
    dimension = model["dimension"]
    l1 = liouvillian(model["H1"], model["collapse"])
    l2 = liouvillian(model["H2"], model["collapse"])
    vector = density_vector_from_bits([0] * parameters.N + [0])
    times = []
    continuous = {name: [] for name in model["observables"]}
    stroboscopic = {name: [] for name in model["observables"]}
    quality = {"trace": [], "hermiticity": [], "minimum_eigenvalue": []}
    current_time = 0.0
    for _ in range(parameters.periods):
        for name, operator in model["observables"].items():
            stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))
        if validate:
            rho = unvec(vector, dimension)
            quality["trace"].append(abs(np.trace(rho) - 1.0))
            quality["hermiticity"].append(np.linalg.norm(rho - rho.conjugate().T))
            quality["minimum_eigenvalue"].append(np.min(np.linalg.eigvalsh(rho)).real)
        trajectory1 = expm_multiply(
            l1, vector, start=0.0, stop=parameters.T1,
            num=parameters.samples_per_step + 1, endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            times.append(current_time + sample * parameters.T1 / parameters.samples_per_step)
            for name, operator in model["observables"].items():
                continuous[name].append(expectation_from_vec(operator, trajectory1[sample], dimension))
        vector = trajectory1[-1]
        current_time += parameters.T1
        trajectory2 = expm_multiply(
            l2, vector, start=0.0, stop=parameters.T2,
            num=parameters.samples_per_step + 1, endpoint=True,
        )
        for sample in range(parameters.samples_per_step):
            times.append(current_time + sample * parameters.T2 / parameters.samples_per_step)
            for name, operator in model["observables"].items():
                continuous[name].append(expectation_from_vec(operator, trajectory2[sample], dimension))
        vector = trajectory2[-1]
        current_time += parameters.T2
    for name, operator in model["observables"].items():
        stroboscopic[name].append(expectation_from_vec(operator, vector, dimension))
    return {
        "parameters": parameters,
        "model": model,
        "time": np.asarray(times),
        "continuous": {name: np.asarray(values) for name, values in continuous.items()},
        "stroboscopic": {name: np.asarray(values) for name, values in stroboscopic.items()},
        "final_vector": vector,
        "quality": {
            "max_trace_error": float(np.max(quality["trace"])) if quality["trace"] else np.nan,
            "max_hermiticity_error": float(np.max(quality["hermiticity"])) if quality["hermiticity"] else np.nan,
            "minimum_eigenvalue": float(np.min(quality["minimum_eigenvalue"])) if quality["minimum_eigenvalue"] else np.nan,
        },
    }


def subharmonic_phasor(time_array, signal, omega, discard_periods, period):
    mask = time_array >= discard_periods * period
    t = time_array[mask]
    y = np.asarray(signal)[mask]
    y = y - np.mean(y)
    return 2.0 * np.trapezoid(y * np.exp(1j * omega * t / 2.0), t) / (t[-1] - t[0])


def metrics(run, discard_periods=20):
    parameters = run["parameters"]
    edge = subharmonic_phasor(
        run["time"], run["continuous"]["edge_left"], parameters.Omega,
        discard_periods, parameters.T,
    )
    tls_lowering = 0.5 * (
        run["continuous"]["tls_x"] + 1j * run["continuous"]["tls_y"]
    )
    tls = subharmonic_phasor(
        run["time"], tls_lowering, parameters.Omega,
        discard_periods, parameters.T,
    )
    late = run["time"] >= discard_periods * parameters.T
    return {
        "A_edge": float(abs(edge)),
        "A_tls": float(abs(tls)),
        "emission": float(parameters.gamma1 * np.mean(run["continuous"]["tls_excited"][late])),
        "relative_phase": float(np.angle(tls / edge)) if abs(edge) > 1e-10 and abs(tls) > 1e-10 else np.nan,
    }


def atomic_savez(path, **arrays):
    path = Path(path)
    temporary = path.with_name(path.name + ".temporary.npz")
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


sanity = simulate_open(replace(p, N=3, periods=6, tls_site=1), validate=True)
print("movable-contact identity error =", sanity["model"]["exchange_identity_error"])
print("sanity quality =", sanity["quality"])
assert sanity["model"]["exchange_identity_error"] < 1e-12
assert sanity["quality"]["max_trace_error"] < 1e-10
assert sanity["quality"]["max_hermiticity_error"] < 1e-10
assert sanity["quality"]["minimum_eigenvalue"] > -1e-10


## 2. Resolved doublet: linear guide versus a threshold model

The four cross-window accepted production points are fitted both to a linear guide and to the minimal resolved-splitting form

\[
s(g)=A\sqrt{g^2-g_c^2},
\qquad
s=\frac{\delta\omega_{\rm resp}}{\Omega/2}.
\]

The purpose is not model selection from four points. It is to test whether the negative intercept of the linear guide is also compatible with a finite linewidth threshold.


In [ ]:
def quadratic_vertex(x_values, y_values, index):
    if index <= 0 or index >= len(x_values) - 1:
        return np.nan, np.nan
    coefficients = np.polyfit(
        x_values[index - 1:index + 2], y_values[index - 1:index + 2], 2
    )
    location = -coefficients[1] / (2.0 * coefficients[0])
    if not (x_values[index - 1] <= location <= x_values[index + 1]):
        location = x_values[index]
    return float(location), float(np.polyval(coefficients, location))


def central_tls_pair(x_values, y_values):
    peaks, _ = find_peaks(
        y_values, prominence=max(1e-8, 0.04 * np.ptp(y_values)), distance=3
    )
    left = [index for index in peaks if 0.86 <= x_values[index] < 1.0]
    right = [index for index in peaks if 1.0 < x_values[index] <= 1.24]
    if not left or not right:
        return None
    left_point = quadratic_vertex(x_values, y_values, max(left, key=lambda index: x_values[index]))
    right_point = quadratic_vertex(x_values, y_values, min(right, key=lambda index: x_values[index]))
    return {
        "half_split": 0.5 * (right_point[0] - left_point[0]),
        "midpoint": 0.5 * (right_point[0] + left_point[0]),
    }


if RUN_THRESHOLD_COMPARISON:
    checkpoint_candidates = [
        Path("floquet_tls_N6_g_frequency_checkpoint.npz"),
        Path("upload/floquet_tls_N6_g_frequency_checkpoint.npz"),
    ]
    checkpoint = next((candidate for candidate in checkpoint_candidates if candidate.exists()), None)
    if checkpoint is None:
        raise FileNotFoundError("N=6 g-frequency checkpoint is required")
    with np.load(checkpoint, allow_pickle=False) as saved:
        production = {key: np.asarray(saved[key]) for key in saved.files}
    g_values = production["g_values"]
    ratios = production["omega_ratios"]
    windows = production["discard_windows"].astype(int)
    split_windows = np.full((len(windows), len(g_values)), np.nan)
    midpoint_windows = np.full_like(split_windows, np.nan)
    for row, discard in enumerate(windows):
        response = production[f"A_tls_transverse_d{discard:02d}"]
        for column in range(len(g_values)):
            pair = central_tls_pair(ratios, response[column])
            if pair is not None:
                split_windows[row, column] = pair["half_split"]
                midpoint_windows[row, column] = pair["midpoint"]
    split_mean = np.full(len(g_values), np.nan)
    split_std = np.full(len(g_values), np.nan)
    midpoint_mean = np.full(len(g_values), np.nan)
    for column in range(len(g_values)):
        valid = np.isfinite(split_windows[:, column])
        if valid.any():
            split_mean[column] = np.mean(split_windows[valid, column])
            midpoint_mean[column] = np.mean(midpoint_windows[valid, column])
        if valid.sum() >= 2:
            split_std[column] = np.std(split_windows[valid, column], ddof=1)
    accepted = (
        np.isfinite(split_windows).all(axis=0)
        & (np.abs(midpoint_mean - 1.0) <= 0.04)
        & (split_std / split_mean <= 0.10)
    )
    g_fit = g_values[accepted]
    split_fit = split_mean[accepted]
    sigma_fit = split_std[accepted]

    linear_coefficients = np.polyfit(g_fit, split_fit, 1)
    linear_prediction = np.polyval(linear_coefficients, g_fit)

    def threshold_model(coupling, amplitude, critical_coupling):
        return amplitude * np.sqrt(np.maximum(coupling ** 2 - critical_coupling ** 2, 0.0))

    threshold_coefficients, threshold_covariance = curve_fit(
        threshold_model, g_fit, split_fit,
        p0=[1.2, 0.04], sigma=sigma_fit, absolute_sigma=True,
        bounds=([0.0, 0.0], [10.0, float(g_fit.min() - 1e-6)]),
    )
    threshold_prediction = threshold_model(g_fit, *threshold_coefficients)

    def r_squared(observed, predicted):
        return 1.0 - np.sum((observed - predicted) ** 2) / np.sum(
            (observed - np.mean(observed)) ** 2
        )

    linear_r2 = r_squared(split_fit, linear_prediction)
    threshold_r2 = r_squared(split_fit, threshold_prediction)
    linear_chi2 = np.sum(((split_fit - linear_prediction) / sigma_fit) ** 2)
    threshold_chi2 = np.sum(((split_fit - threshold_prediction) / sigma_fit) ** 2)

    dense_g = np.linspace(g_fit.min(), g_fit.max(), 300)
    fig, axis = plt.subplots(figsize=(7.2, 4.6))
    axis.errorbar(g_fit, split_fit, yerr=sigma_fit, fmt="ko", capsize=3, label="accepted production points")
    axis.plot(dense_g, np.polyval(linear_coefficients, dense_g), "--", label=fr"linear guide, $R^2={linear_r2:.4f}$")
    axis.plot(dense_g, threshold_model(dense_g, *threshold_coefficients), "-", label=fr"threshold model, $R^2={threshold_r2:.4f}$")
    axis.axvline(threshold_coefficients[1], color="0.4", ls=":", label=fr"$g_c={threshold_coefficients[1]:.4f}$")
    axis.set(xlabel=r"bare coupling $g$", ylabel=r"response half-splitting / $(\Omega/2)$", title="Two descriptions are indistinguishable with four resolved points")
    axis.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    print("accepted couplings =", g_fit)
    print("linear [slope, intercept] =", linear_coefficients, "R2 =", linear_r2, "chi2 =", linear_chi2)
    print("threshold [A, gc] =", threshold_coefficients, "R2 =", threshold_r2, "chi2 =", threshold_chi2)
else:
    print("threshold comparison skipped")


## 3. Closed-system dressed (0-π) matrix element

The chain-only many-body Floquet operator is diagonalized at (N=6). For Floquet modes (|u_a(t)\rangle),

\[
B_{ab}^{(m)}(j)=\frac1T\int_0^T dt\,
e^{-im\Omega t}\langle u_a(t)|s_-^j|u_b(t)\rangle.
\]

Among quasienergy pairs separated by π within a stated tolerance, the pair with the largest boundary (m=0) matrix element is selected. Its full spatial profile is then fitted to a two-edge form; the pair is never reselected separately at each site.


In [ ]:
def dense_kron_all(factors):
    output = np.array([[1.0 + 0.0j]])
    for factor in factors:
        output = np.kron(output, factor)
    return output


def dense_site_operator(local_operator, site, n_total):
    return dense_kron_all([
        local_operator if index == site else np.eye(2, dtype=complex)
        for index in range(n_total)
    ])


def closed_floquet_sidebands(parameters, sidebands=(-1, 0, 1), time_points=81):
    N = parameters.N
    x = np.array([[0, 1], [1, 0]], dtype=complex)
    z = np.diag([1.0, -1.0]).astype(complex)
    sm = np.array([[0, 1], [0, 0]], dtype=complex)
    x_ops = [dense_site_operator(x, site, N) for site in range(N)]
    z_ops = [dense_site_operator(z, site, N) for site in range(N)]
    sm_ops = [dense_site_operator(sm, site, N) for site in range(N)]
    h1 = sum(
        (-parameters.J * z_ops[site] @ z_ops[site + 1] for site in range(N - 1)),
        np.zeros_like(x_ops[0]),
    )
    h2 = sum((-parameters.h * operator for operator in x_ops), np.zeros_like(x_ops[0]))
    e1, v1 = np.linalg.eigh(h1)
    e2, v2 = np.linalg.eigh(h2)

    def propagator(eigenvalues, eigenvectors, duration):
        return (eigenvectors * np.exp(-1j * eigenvalues * duration)) @ eigenvectors.conjugate().T

    u1 = propagator(e1, v1, parameters.T1)
    u2 = propagator(e2, v2, parameters.T2)
    floquet = u2 @ u1
    triangular, floquet_vectors = schur(floquet, output="complex")
    triangular_error = np.linalg.norm(triangular - np.diag(np.diag(triangular)))
    eigenvalues = np.diag(triangular)
    phases = np.angle(eigenvalues)
    quasienergies = -phases / parameters.T
    times = np.linspace(0.0, parameters.T, time_points)
    sideband_matrices = {
        m: np.zeros((N, len(eigenvalues), len(eigenvalues)), dtype=complex)
        for m in sidebands
    }
    for time_index, current_time in enumerate(times):
        if current_time <= parameters.T1 + 1e-14:
            evolution = propagator(e1, v1, current_time)
        else:
            evolution = propagator(e2, v2, current_time - parameters.T1) @ u1
        floquet_modes = (
            evolution @ floquet_vectors
            * np.exp(1j * quasienergies * current_time)[None, :]
        )
        trapezoid_weight = 0.5 if time_index in (0, len(times) - 1) else 1.0
        for site, operator in enumerate(sm_ops):
            instantaneous = floquet_modes.conjugate().T @ operator @ floquet_modes
            for m in sidebands:
                sideband_matrices[m][site] += (
                    trapezoid_weight
                    * np.exp(-1j * m * parameters.Omega * current_time)
                    * instantaneous
                )
    dt = times[1] - times[0]
    for m in sidebands:
        sideband_matrices[m] *= dt / parameters.T
    return {
        "phases": phases,
        "quasienergies": quasienergies,
        "sidebands": sideband_matrices,
        "schur_offdiagonal_error": triangular_error,
    }


def select_pi_pair(data, phase_tolerance=0.02, site=0, sideband=0):
    phases = data["phases"]
    matrix = data["sidebands"][sideband][site]
    candidates = []
    for first in range(len(phases)):
        for second in range(first + 1, len(phases)):
            phase_error = abs(np.angle(np.exp(1j * (phases[second] - phases[first] - np.pi))))
            amplitude = max(abs(matrix[first, second]), abs(matrix[second, first]))
            if phase_error <= phase_tolerance:
                candidates.append((amplitude, phase_error, first, second))
    if not candidates:
        raise RuntimeError("no pi-separated Floquet pair found")
    return max(candidates, key=lambda row: row[0]), candidates


def majorana_generators(N, J=1.0, h=1.0):
    a_zz = np.zeros((2 * N, 2 * N), dtype=float)
    a_x = np.zeros((2 * N, 2 * N), dtype=float)
    for site in range(N):
        a_x[2 * site, 2 * site + 1] = -2.0 * h
        a_x[2 * site + 1, 2 * site] = 2.0 * h
    for site in range(N - 1):
        a_zz[2 * site + 1, 2 * site + 2] = -2.0 * J
        a_zz[2 * site + 2, 2 * site + 1] = 2.0 * J
    return a_zz, a_x


def pi_majorana_finite_size_length(parameters, sizes=np.arange(2, 33)):
    splittings = []
    for chain_size in sizes:
        a_zz, a_x = majorana_generators(
            int(chain_size), parameters.J, parameters.h
        )
        rotation = (
            expm(a_x * parameters.T2)
            @ expm(a_zz * parameters.T1)
        )
        phases = np.angle(np.linalg.eigvals(rotation))
        splittings.append(
            np.min(np.abs(np.angle(-np.exp(1j * phases))))
        )
    splittings = np.asarray(splittings)
    resolved = (splittings > 1e-13) & np.isfinite(splittings)
    slope, intercept = np.polyfit(sizes[resolved], np.log(splittings[resolved]), 1)
    return -1.0 / slope, splittings, resolved


if RUN_DRESSED_MATRIX_ELEMENT:
    dressed_start = time.perf_counter()
    dressed_parameters = replace(p, N=6)
    dressed_data = closed_floquet_sidebands(dressed_parameters)
    selected_pair, pi_candidates = select_pi_pair(dressed_data)
    boundary_amplitude, pair_phase_error, pair_a, pair_b = selected_pair
    profiles = {}
    for m, matrices in dressed_data["sidebands"].items():
        profiles[m] = np.maximum(
            np.abs(matrices[:, pair_a, pair_b]),
            np.abs(matrices[:, pair_b, pair_a]),
        )
    profile = profiles[0]
    sites = np.arange(dressed_parameters.N)

    def two_edge_profile(site_values, amplitude, localization_length):
        return amplitude * (
            np.exp(-site_values / localization_length)
            + np.exp(-(dressed_parameters.N - 1 - site_values) / localization_length)
        )

    spatial_coefficients, spatial_covariance = curve_fit(
        two_edge_profile, sites, profile / profile.max(),
        p0=[1.0, 0.35], bounds=([0.0, 0.02], [10.0, 10.0]),
    )
    xi_dressed = spatial_coefficients[1]
    xi_pi_majorana, pi_splittings, pi_resolved = pi_majorana_finite_size_length(
        dressed_parameters
    )

    time_grid_convergence = {}
    reference_normalized_profile = profile / profile.max()
    for time_points in [41, 81, 161]:
        convergence_data = closed_floquet_sidebands(
            dressed_parameters, time_points=time_points
        )
        convergence_pair, _ = select_pi_pair(convergence_data)
        _, _, convergence_a, convergence_b = convergence_pair
        convergence_profile = np.maximum(
            np.abs(convergence_data["sidebands"][0][:, convergence_a, convergence_b]),
            np.abs(convergence_data["sidebands"][0][:, convergence_b, convergence_a]),
        )
        convergence_profile = convergence_profile / convergence_profile.max()
        convergence_fit, _ = curve_fit(
            two_edge_profile, sites, convergence_profile,
            p0=[1.0, 0.35], bounds=([0.0, 0.02], [10.0, 10.0]),
        )
        time_grid_convergence[time_points] = {
            "xi_B": float(convergence_fit[1]),
            "max_profile_change_from_81": float(
                np.max(np.abs(convergence_profile - reference_normalized_profile))
            ),
        }

    tolerance_convergence = {}
    for tolerance in [0.006, 0.010, 0.020, 0.050]:
        tolerance_pair, _ = select_pi_pair(
            dressed_data, phase_tolerance=tolerance
        )
        tolerance_convergence[tolerance] = {
            "amplitude": float(tolerance_pair[0]),
            "phase_error": float(tolerance_pair[1]),
            "pair": (int(tolerance_pair[2]), int(tolerance_pair[3])),
        }

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
    for m, marker in zip([-1, 0, 1], ["^", "o", "s"]):
        axes[0].semilogy(
            sites, profiles[m] / max(profiles[m].max(), 1e-16),
            marker + "-", label=fr"$m={m}$",
        )
    axes[0].plot(
        sites, two_edge_profile(sites, *spatial_coefficients), "k--",
        label=fr"two-edge fit for $m=0$, $\xi_B={xi_dressed:.3f}$",
    )
    axes[0].set(xlabel="chain site", ylabel="normalized dressed matrix element", title="The selected pair is localized at both boundaries")
    axes[0].legend(fontsize=8)

    candidate_amplitudes = np.asarray([row[0] for row in pi_candidates])
    candidate_errors = np.asarray([row[1] for row in pi_candidates])
    axes[1].scatter(candidate_errors, candidate_amplitudes, s=18, alpha=0.55)
    axes[1].scatter(pair_phase_error, boundary_amplitude, s=70, marker="*", color="C3", label="selected pair")
    axes[1].set(xlabel="circular phase error from pi", ylabel=r"boundary $|B_{ab}^{(0)}|$", title="Pair selection uses the boundary matrix element")
    axes[1].legend()
    fig.tight_layout()
    plt.show()

    print("dressed runtime (s) =", time.perf_counter() - dressed_start)
    print("Schur off-diagonal error =", dressed_data["schur_offdiagonal_error"])
    print("selected pair =", (pair_a, pair_b), "phase error =", pair_phase_error)
    print("m=0 profile =", profile)
    print("normalized m=0 profile =", profile / profile.max())
    print("two-edge localization length xi_B =", xi_dressed)
    print("independent Majorana finite-size length xi_pi =", xi_pi_majorana)
    print("time-grid convergence =", time_grid_convergence)
    print("pair-tolerance convergence =", tolerance_convergence)
    print("edge sideband amplitudes =", {m: profiles[m][0] for m in profiles})
else:
    print("dressed matrix-element calculation skipped")


## 4. Reduced channel and exact (N=3) channel benchmark

A single selected Floquet doublet plus the TLS defines a four-state reduced model. Its edge coherence is read out with (X_e), so the uncoupled channel has an eigenvalue at (-1). This model is compared with the exact (N=3) chain+TLS channel at the same central detuning.

Agreement is not assumed. A discrepancy would show that the observed response involves a many-doublet manifold or observable-weighted collective coupling rather than one isolated many-body pair.


In [ ]:
def channel_visibility(channel, initial_vector, readout_vector):
    eigenvalues, right_vectors = np.linalg.eig(channel)
    coefficients = np.linalg.solve(right_vectors, initial_vector)
    readout = readout_vector.conjugate() @ right_vectors
    visibility = np.abs(readout * coefficients)
    phase_offset = np.angle(np.exp(1j * (np.angle(eigenvalues) - np.pi)))
    return {
        "eigenvalues": eigenvalues,
        "right_vectors": right_vectors,
        "visibility": visibility,
        "phase_offset_over_T": phase_offset / p.T,
    }


def reduced_channel(coupling, dressed_factor, gamma1=0.08, detuning=0.0):
    identity = np.eye(2, dtype=complex)
    x = np.array([[0, 1], [1, 0]], dtype=complex)
    y = np.array([[0, -1j], [1j, 0]], dtype=complex)
    z = np.diag([1.0, -1.0]).astype(complex)
    sm = np.array([[0, 1], [0, 0]], dtype=complex)
    sp = sm.conjugate().T
    edge_z = np.kron(z, identity)
    edge_x = np.kron(x, identity)
    tls_z = np.kron(identity, z)
    tls_sm = np.kron(identity, sm)
    hamiltonian = (
        -0.25 * p.Omega * edge_z
        -0.5 * (p.Omega / 2.0 + detuning) * tls_z
        + coupling * dressed_factor * (
            np.kron(sp, sm) + np.kron(sm, sp)
        )
    )
    collapse = [np.sqrt(gamma1) * tls_sm]
    generator = liouvillian(csr_matrix(hamiltonian), [csr_matrix(item) for item in collapse]).toarray()
    channel = expm(generator * p.T)
    plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2.0)
    ground = np.array([1.0, 0.0], dtype=complex)
    ket = np.kron(plus, ground)
    initial = np.outer(ket, ket.conjugate()).reshape(-1, order="F")
    readout = edge_x.reshape(-1, order="F")
    return channel_visibility(channel, initial, readout)


def full_channel_spectrum(parameters):
    model = build_open_model(parameters)
    l1 = liouvillian(model["H1"], model["collapse"]).toarray()
    l2 = liouvillian(model["H2"], model["collapse"]).toarray()
    channel = expm(l2 * parameters.T2) @ expm(l1 * parameters.T1)
    initial = density_vector_from_bits([0] * parameters.N + [0])
    readout = model["observables"]["edge_left"].toarray().reshape(-1, order="F")
    return channel_visibility(channel, initial, readout)


def track_conjugate_pi_pair(spectra, coupling_values, offset_limit=0.20):
    descending = coupling_values[::-1]
    tracked = {1: [], -1: []}
    first_spectrum = spectra[float(descending[0])]
    for sign in [1, -1]:
        offset = first_spectrum["phase_offset_over_T"]
        candidates = np.where((sign * offset > 1e-6) & (sign * offset < offset_limit))[0]
        selected = candidates[np.argmax(first_spectrum["visibility"][candidates])]
        tracked[sign].append({
            "g": float(descending[0]), "index": int(selected),
            "vector": first_spectrum["right_vectors"][:, selected],
            "offset": float(offset[selected]),
            "visibility": float(first_spectrum["visibility"][selected]),
            "overlap": 1.0,
        })
    for coupling in descending[1:]:
        spectrum = spectra[float(coupling)]
        for sign in [1, -1]:
            previous = tracked[sign][-1]["vector"]
            offset = spectrum["phase_offset_over_T"]
            candidates = np.where((sign * offset > 0.0) & (sign * offset < offset_limit))[0]
            overlaps = np.abs(previous.conjugate() @ spectrum["right_vectors"][:, candidates])
            selected = candidates[np.argmax(overlaps)]
            tracked[sign].append({
                "g": float(coupling), "index": int(selected),
                "vector": spectrum["right_vectors"][:, selected],
                "offset": float(offset[selected]),
                "visibility": float(spectrum["visibility"][selected]),
                "overlap": float(np.max(overlaps)),
            })
    half_split = np.full(len(coupling_values), np.nan)
    visibility = np.full(len(coupling_values), np.nan)
    modulus = np.full(len(coupling_values), np.nan)
    for index, coupling in enumerate(coupling_values):
        positive = next(row for row in tracked[1] if np.isclose(row["g"], coupling))
        negative = next(row for row in tracked[-1] if np.isclose(row["g"], coupling))
        half_split[index] = 0.5 * (positive["offset"] - negative["offset"])
        visibility[index] = min(positive["visibility"], negative["visibility"])
        eigenvalue = spectra[float(coupling)]["eigenvalues"][positive["index"]]
        modulus[index] = abs(eigenvalue)
    return half_split, visibility, modulus, tracked


if RUN_REDUCED_AND_N3_CHANNEL:
    channel_start = time.perf_counter()
    channel_g = np.asarray([0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.16])
    selected_dressed_factor = float(boundary_amplitude)
    reduced_spectra = {
        float(coupling): reduced_channel(coupling, selected_dressed_factor)
        for coupling in channel_g
    }
    exact_n3_spectra = {
        float(coupling): full_channel_spectrum(
            replace(p, N=3, g=float(coupling), periods=8, tls_site=0)
        )
        for coupling in channel_g
    }
    exact_n4_spectra = (
        {
            float(coupling): full_channel_spectrum(
                replace(p, N=4, g=float(coupling), periods=8, tls_site=0)
            )
            for coupling in channel_g
        }
        if RUN_EXACT_N4_CHANNEL else None
    )
    reduced_split, reduced_visibility, reduced_modulus, reduced_tracked = track_conjugate_pi_pair(
        reduced_spectra, channel_g, offset_limit=0.35
    )
    exact_n3_split, exact_n3_visibility, exact_n3_modulus, exact_n3_tracked = track_conjugate_pi_pair(
        exact_n3_spectra, channel_g, offset_limit=0.20
    )
    if exact_n4_spectra is not None:
        exact_n4_split, exact_n4_visibility, exact_n4_modulus, exact_n4_tracked = track_conjugate_pi_pair(
            exact_n4_spectra, channel_g, offset_limit=0.20
        )

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
    axes[0].plot(channel_g, reduced_split, "o-", label="single-doublet reduced channel")
    axes[0].plot(channel_g, exact_n3_split, "s-", label="exact N=3 channel")
    if exact_n4_spectra is not None:
        axes[0].plot(channel_g, exact_n4_split, "^-", label="exact N=4 channel")
    response_half_split_frequency = split_fit * p.Omega / 2.0
    response_half_split_error = sigma_fit * p.Omega / 2.0
    axes[0].errorbar(
        g_fit, response_half_split_frequency,
        yerr=response_half_split_error, fmt="ko", capsize=3,
        label="N=6 response doublet",
    )
    axes[0].set(xlabel="bare coupling g", ylabel="half splitting (frequency units)", title="Small channels do not yet predict the production response splitting")
    axes[0].legend(fontsize=8)
    axes[1].semilogy(channel_g, reduced_visibility, "o-", label="reduced")
    axes[1].semilogy(channel_g, exact_n3_visibility, "s-", label="exact N=3")
    if exact_n4_spectra is not None:
        axes[1].semilogy(channel_g, exact_n4_visibility, "^-", label="exact N=4")
    axes[1].set(xlabel="bare coupling g", ylabel="minimum tracked visibility", title="Weak-coupling branches become difficult to observe")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    print("channel runtime (s) =", time.perf_counter() - channel_start)
    print("selected single-pair dressed factor =", selected_dressed_factor)
    print("reduced half splittings =", dict(zip(channel_g, reduced_split)))
    print("exact N=3 half splittings =", dict(zip(channel_g, exact_n3_split)))
    if exact_n4_spectra is not None:
        print("exact N=4 half splittings =", dict(zip(channel_g, exact_n4_split)))
    print("N=6 response half splittings (frequency units) =", dict(zip(g_fit, response_half_split_frequency)))
    print("reduced moduli =", dict(zip(channel_g, reduced_modulus)))
    print("exact N=3 moduli =", dict(zip(channel_g, exact_n3_modulus)))
    if exact_n4_spectra is not None:
        print("exact N=4 moduli =", dict(zip(channel_g, exact_n4_modulus)))
else:
    print("reduced and N=3 channel benchmarks skipped")


## 5. (N=4) position--frequency pilot

The affordable open-system pilot scans all four TLS contact positions and 31 frequencies. It tests reflection symmetry, peak structure, maximum response, and integrated spectral weight. Because both boundaries are present, a boundary-localized response should be large at (j=0,3) and suppressed at (j=1,2); a monotonic one-edge exponential is not expected on this short chain.


In [ ]:
def initialize_position_checkpoint(path, sites, ratios):
    path = Path(path)
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            data = {key: np.asarray(saved[key]) for key in saved.files}
        if not np.array_equal(data["sites"], sites) or not np.allclose(data["ratios"], ratios):
            raise ValueError("position checkpoint grid mismatch")
        return data
    return {
        "sites": sites,
        "ratios": ratios,
        "completed": np.zeros((len(sites), len(ratios)), dtype=bool),
        "A_edge": np.full((len(sites), len(ratios)), np.nan),
        "A_tls": np.full((len(sites), len(ratios)), np.nan),
        "emission": np.full((len(sites), len(ratios)), np.nan),
        "relative_phase": np.full((len(sites), len(ratios)), np.nan),
        "metadata_N": np.asarray(4),
        "metadata_periods": np.asarray(60),
        "metadata_samples_per_step": np.asarray(2),
        "metadata_g": np.asarray(0.08),
        "metadata_gamma1": np.asarray(0.08),
    }


if RUN_N4_POSITION_PILOT:
    position_start = time.perf_counter()
    position_path = Path("floquet_tls_N4_position_frequency_pilot.npz")
    position_sites = np.arange(4, dtype=int)
    position_ratios = np.linspace(0.80, 1.20, 31)
    position_data = initialize_position_checkpoint(
        position_path, position_sites, position_ratios
    )
    for site_index, contact_site in enumerate(position_sites):
        for ratio_index, ratio in enumerate(position_ratios):
            if position_data["completed"][site_index, ratio_index]:
                continue
            parameters = replace(
                p, N=4, tls_site=int(contact_site), g=0.08, gamma1=0.08,
                omega_d=float(ratio * p.Omega / 2.0), periods=60,
                samples_per_step=2,
            )
            point_run = simulate_open(parameters)
            point_metrics = metrics(point_run, discard_periods=20)
            for key in ["A_edge", "A_tls", "emission", "relative_phase"]:
                position_data[key][site_index, ratio_index] = point_metrics[key]
            position_data["completed"][site_index, ratio_index] = True
            atomic_savez(position_path, **position_data)

    tls_maximum = np.max(position_data["A_tls"], axis=1)
    emission_maximum = np.max(position_data["emission"], axis=1)
    tls_weight = np.trapezoid(position_data["A_tls"] ** 2, position_ratios, axis=1)
    reflection_errors = {
        "A_tls": float(np.max(np.abs(position_data["A_tls"] - position_data["A_tls"][::-1]))),
        "emission": float(np.max(np.abs(position_data["emission"] - position_data["emission"][::-1]))),
    }

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))
    image = axes[0].pcolormesh(
        position_ratios, position_sites, position_data["A_tls"],
        shading="auto", cmap="viridis",
    )
    axes[0].set(xlabel=r"$\omega_d/(\Omega/2)$", ylabel="TLS contact site", yticks=position_sites, title="TLS transverse response")
    fig.colorbar(image, ax=axes[0], label=r"$A_{TLS}^{\perp}$")
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(position_sites)))
    for site_index, (contact_site, color) in enumerate(zip(position_sites, colors)):
        axes[1].plot(position_ratios, position_data["A_tls"][site_index], color=color, label=f"site {contact_site}")
    axes[1].set(xlabel=r"$\omega_d/(\Omega/2)$", ylabel=r"$A_{TLS}^{\perp}$", title="Full spectra reveal two-boundary symmetry")
    axes[1].legend(fontsize=8)
    axes[2].plot(position_sites, tls_maximum / tls_maximum.max(), "o-", label="maximum TLS amplitude")
    axes[2].plot(position_sites, tls_weight / tls_weight.max(), "s-", label="integrated TLS spectral weight")
    axes[2].plot(position_sites, emission_maximum / emission_maximum.max(), "^-", label="maximum emission")
    axes[2].set(xlabel="TLS contact site", ylabel="normalized response", xticks=position_sites, ylim=(0, 1.08), title="Raw observables need not follow one matrix element")
    axes[2].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    print("position pilot runtime (s) =", time.perf_counter() - position_start)
    print("completion =", int(position_data["completed"].sum()), "/", position_data["completed"].size)
    print("TLS maximum by site =", dict(zip(position_sites, tls_maximum)))
    print("integrated TLS weight by site =", dict(zip(position_sites, tls_weight)))
    print("emission maximum by site =", dict(zip(position_sites, emission_maximum)))
    print("boundary/bulk TLS-maximum ratio =", float(tls_maximum[0] / tls_maximum[1]))
    print("boundary/bulk integrated-weight ratio =", float(tls_weight[0] / tls_weight[1]))
    print("boundary/bulk emission-maximum ratio =", float(emission_maximum[0] / emission_maximum[1]))
    print("reflection errors =", reflection_errors)
else:
    print("N=4 position pilot skipped")


## 6. Results of the affordable validation layer

- The four resolved production points are consistent with both the linear guide and a threshold form. The threshold fit gives \(g_c=0.0407\), precisely between the unresolved \(g=0.04\) point and the first fully resolved \(g=0.06\) point. Four points do not select one model.
- For the fixed \(N=6\) \(0-\pi\) pair, the \(m=0\) dressed profile is
  
  \[
  |B(j)|/|B(0)|=(1,0.0499,0.0030,0.0030,0.0499,1).
  \]
  
  Its two-edge length is \(\xi_B=0.334\), close to the independently recomputed Majorana finite-size length. Time-grid and pair-tolerance tests are printed above and must remain stable before this becomes a paper claim.
- The single-doublet reduced channel develops a visible split only after its damping threshold. Exact \(N=3\) and \(N=4\) channels give larger, visible branches, but all three channel benchmarks remain below the \(N=6\) response-doublet splitting. Therefore one selected many-body pair and the present small-system channels are not yet quantitative reductions of the production observable. A multi-doublet spectral projector or a production-size channel remains necessary.
- The \(N=4\) open-system position pilot is exactly reflection symmetric. Boundary contacts enhance the integrated TLS spectral weight by approximately \(3.26\) and the maximum emission by approximately \(1.94\) relative to the inner contacts. The observable enhancement is much weaker than the selected-pair matrix-element decay, confirming that raw response includes multiple Floquet channels and peak motion.

The affordable layer therefore establishes **boundary selectivity**, but not yet the quantitative identity \(\Delta\omega_{\rm resp}=2|gB_{0\pi}^{(0)}|\).


## 7. Disabled local production cells

The following functions are included for checkpointed local execution. They are disabled here because their estimated cell runtime exceeds five minutes.

- The (N=6) position grid uses all six contact sites, 61 frequencies, 80 periods, four samples per drive step, and discard windows (8,20,40).
- The (N=6) matrix-free channel never constructs the dense (16384\times16384) superoperator. It uses the filtered action \(A_\pi=(I-\mathcal F)/2\), an edge-seeded **unrestarted** Arnoldi basis, a fixed Krylov budget, and an explicit Ritz residual. The old restarted `eigs(..., which="LM")` calculation is intentionally not used because it spends most of its work on irrelevant large-modulus modes and can remain silent for many minutes.


In [ ]:
POSITION_CHECKPOINT_STEM = "floquet_tls_N6_position_frequency_checkpoint"


def _new_long_position_checkpoint(sites, ratios, windows):
    data = {
        "sites": sites,
        "ratios": ratios,
        "windows": windows,
        "completed": np.zeros((len(sites), len(ratios)), dtype=bool),
        "metadata_N": np.asarray(6),
        "metadata_periods": np.asarray(80),
        "metadata_samples_per_step": np.asarray(4),
        "metadata_g": np.asarray(0.08),
        "metadata_gamma1": np.asarray(0.08),
    }
    for discard in windows:
        for metric_name in ["A_edge", "A_tls", "emission", "relative_phase"]:
            data[f"{metric_name}_d{discard:02d}"] = np.full(
                (len(sites), len(ratios)), np.nan
            )
    return data


def _load_and_validate_position_checkpoint(path, sites, ratios, windows):
    """Load one candidate, rejecting stale/corrupt/incompatible grids."""
    path = Path(path)
    with np.load(path, allow_pickle=False) as saved:
        data = {key: np.asarray(saved[key]) for key in saved.files}

    required = {"sites", "ratios", "windows", "completed"}
    for discard in windows:
        required.update(
            f"{metric_name}_d{discard:02d}"
            for metric_name in ["A_edge", "A_tls", "emission", "relative_phase"]
        )
    missing = sorted(required.difference(data))
    if missing:
        raise ValueError(f"missing keys: {missing}")
    if not np.array_equal(np.asarray(data["sites"], dtype=int), sites):
        raise ValueError("site grid mismatch")
    if data["ratios"].shape != ratios.shape or not np.allclose(data["ratios"], ratios):
        raise ValueError("frequency grid mismatch")
    if not np.array_equal(np.asarray(data["windows"], dtype=int), windows):
        raise ValueError("discard-window grid mismatch")

    expected_shape = (len(sites), len(ratios))
    if data["completed"].shape != expected_shape:
        raise ValueError(
            f"completed mask has shape {data['completed'].shape}, expected {expected_shape}"
        )
    data["completed"] = np.asarray(data["completed"], dtype=bool)
    for key in required.difference({"sites", "ratios", "windows", "completed"}):
        if data[key].shape != expected_shape:
            raise ValueError(f"{key} has shape {data[key].shape}, expected {expected_shape}")
    return data


def resolve_long_position_checkpoint(directory, sites, ratios, windows):
    """
    Select the compatible checkpoint with the most completed points.

    This deliberately recognizes browser-renamed copies such as ``...(1).npz``.
    If candidate files exist but none match the current grid, it stops instead of
    silently replacing them with a blank run.
    """
    directory = Path(directory).expanduser().resolve()
    directory.mkdir(parents=True, exist_ok=True)
    exact_path = directory / f"{POSITION_CHECKPOINT_STEM}.npz"
    candidates = sorted(
        candidate for candidate in directory.glob(f"{POSITION_CHECKPOINT_STEM}*.npz")
        if ".temporary" not in candidate.name
    )

    valid = []
    rejected = []
    for candidate in candidates:
        try:
            data = _load_and_validate_position_checkpoint(
                candidate, sites, ratios, windows
            )
        except Exception as error:
            rejected.append((candidate, str(error)))
            continue
        completed_count = int(np.count_nonzero(data["completed"]))
        valid.append((completed_count, candidate == exact_path, candidate, data))

    print("checkpoint search directory:", directory)
    for completed_count, _, candidate, _ in valid:
        print(
            f"  valid: {candidate.name} | "
            f"completed={completed_count}/{len(sites) * len(ratios)}"
        )
    for candidate, reason in rejected:
        print(f"  rejected: {candidate.name} | {reason}")

    if valid:
        # Most completed wins. An exact-name file wins only when completion ties.
        completed_count, _, selected_path, data = max(
            valid, key=lambda item: (item[0], item[1], item[2].stat().st_mtime)
        )
        print("selected checkpoint:", selected_path)
        remaining = np.argwhere(~data["completed"])
        if len(remaining):
            site_index, ratio_index = remaining[0]
            print(
                "resume point: "
                f"site={int(sites[site_index])}, "
                f"ratio={float(ratios[ratio_index]):.4f} "
                f"({completed_count}/{len(sites) * len(ratios)} already complete)"
            )
        else:
            print("checkpoint grid is already complete")
        return selected_path, data

    if candidates:
        raise ValueError(
            "Checkpoint-like files were found, but none match this production grid. "
            "No blank checkpoint was created; inspect the rejected-file messages above."
        )

    data = _new_long_position_checkpoint(sites, ratios, windows)
    print("no compatible checkpoint found; a new run would use:", exact_path)
    return exact_path, data


def inspect_long_position_checkpoints():
    """Print selection and the first missing point without running a simulation."""
    sites = np.arange(6, dtype=int)
    ratios = np.linspace(0.75, 1.25, 61)
    windows = np.asarray([8, 20, 40], dtype=int)
    path, data = resolve_long_position_checkpoint(
        CHECKPOINT_DIRECTORY, sites, ratios, windows
    )
    return path, data


def run_long_n6_position_grid():
    sites = np.arange(6, dtype=int)
    ratios = np.linspace(0.75, 1.25, 61)
    windows = np.asarray([8, 20, 40], dtype=int)
    path, data = resolve_long_position_checkpoint(
        CHECKPOINT_DIRECTORY, sites, ratios, windows
    )
    for site_index, contact_site in enumerate(sites):
        for ratio_index, ratio in enumerate(ratios):
            if data["completed"][site_index, ratio_index]:
                continue
            point_start = time.perf_counter()
            parameters = replace(
                p, N=6, tls_site=int(contact_site), g=0.08, gamma1=0.08,
                omega_d=float(ratio * p.Omega / 2.0), periods=80,
                samples_per_step=4,
            )
            point_run = simulate_open(parameters)
            for discard in windows:
                point_metrics = metrics(point_run, discard_periods=int(discard))
                for metric_name, value in point_metrics.items():
                    data[f"{metric_name}_d{discard:02d}"][site_index, ratio_index] = value
            data["completed"][site_index, ratio_index] = True
            atomic_savez(path, **data)
            print(
                f"site={contact_site}, ratio={ratio:.4f}, "
                f"runtime={time.perf_counter() - point_start:.1f}s"
            )
    return data


def floquet_channel_action(parameters):
    """Return an optimized one-period action without materializing the channel."""
    model = build_open_model(parameters)
    l1_step = (liouvillian(model["H1"], model["collapse"]) * parameters.T1).tocsr()
    l2_step = (liouvillian(model["H2"], model["collapse"]) * parameters.T2).tocsr()
    trace_l1 = l1_step.diagonal().sum()
    trace_l2 = l2_step.diagonal().sum()

    def action(vector):
        after_h1 = expm_multiply(l1_step, np.asarray(vector), traceA=trace_l1)
        return expm_multiply(l2_step, after_h1, traceA=trace_l2)

    return action, model


def analyze_pi_ritz(hessenberg, used_dimension, period, phase_window=0.20):
    """Analyze Ritz modes of A_pi=(I-F)/2 and residuals for F."""
    projected = hessenberg[:used_dimension, :used_dimension]
    mu, projected_vectors = eig(projected)
    eigenvalues = 1.0 - 2.0 * mu
    residuals = (
        2.0 * np.abs(
            hessenberg[used_dimension, used_dimension - 1]
            * projected_vectors[-1, :]
        )
        / np.linalg.norm(projected_vectors, axis=0)
    )
    phase_offsets = np.angle(
        np.exp(1j * (np.angle(eigenvalues) - np.pi))
    ) / period
    edge_readout = np.abs(projected_vectors[0, :])
    selected = {}
    for sign in [1, -1]:
        candidates = np.where(
            (sign * phase_offsets > 1e-8)
            & (sign * phase_offsets < phase_window)
            & (np.abs(eigenvalues) <= 1.0 + 5.0 * residuals)
        )[0]
        if len(candidates) == 0:
            selected[sign] = None
        else:
            score = edge_readout[candidates] / (residuals[candidates] + 1e-14)
            selected[sign] = int(candidates[np.argmax(score)])
    return {
        "eigenvalues": eigenvalues,
        "residuals": residuals,
        "phase_offsets_over_T": phase_offsets,
        "edge_readout": edge_readout,
        "selected": selected,
    }


def edge_seeded_pi_arnoldi(
    parameters, krylov_dim=200, report_every=20,
    residual_tol=1e-3, phase_window=0.20,
):
    """Fixed-budget unrestarted Arnoldi targeted at edge-visible pi modes."""
    channel_action, model = floquet_channel_action(parameters)
    edge_matrix = model["observables"]["edge_left"].toarray().astype(complex)
    edge_matrix -= (
        np.trace(edge_matrix) / model["dimension"]
    ) * np.eye(model["dimension"])
    seed = edge_matrix.reshape(-1, order="F")
    seed /= np.linalg.norm(seed)

    dimension_liouville = seed.size
    estimated_memory_mb = dimension_liouville * (krylov_dim + 1) * 16 / 1024**2
    print(
        f"N={parameters.N}, g={parameters.g:.3f}: Liouville dimension="
        f"{dimension_liouville}, Arnoldi basis approximately "
        f"{estimated_memory_mb:.1f} MiB"
    )
    basis = np.zeros((dimension_liouville, krylov_dim + 1), dtype=complex)
    hessenberg = np.zeros((krylov_dim + 1, krylov_dim), dtype=complex)
    basis[:, 0] = seed
    history = []
    started = time.perf_counter()
    used_dimension = 0

    for column in range(krylov_dim):
        propagated = channel_action(basis[:, column])
        vector = 0.5 * (basis[:, column] - propagated)
        for _ in range(2):
            coefficients = basis[:, :column + 1].conjugate().T @ vector
            hessenberg[:column + 1, column] += coefficients
            vector -= basis[:, :column + 1] @ coefficients
        hessenberg[column + 1, column] = np.linalg.norm(vector)
        used_dimension = column + 1
        if hessenberg[column + 1, column] < 1e-13:
            print(f"Arnoldi breakdown after dimension {used_dimension}")
            break
        basis[:, column + 1] = vector / hessenberg[column + 1, column]

        if used_dimension % report_every == 0 or used_dimension == krylov_dim:
            analysis = analyze_pi_ritz(
                hessenberg, used_dimension, parameters.T,
                phase_window=phase_window,
            )
            report_row = [float(parameters.g), float(used_dimension)]
            printable = []
            for sign in [1, -1]:
                index = analysis["selected"][sign]
                if index is None:
                    report_row.extend([np.nan] * 5)
                    printable.append(f"sign={sign:+d}: none")
                    continue
                eigenvalue = analysis["eigenvalues"][index]
                residual = analysis["residuals"][index]
                offset = analysis["phase_offsets_over_T"][index]
                readout = analysis["edge_readout"][index]
                report_row.extend([
                    eigenvalue.real, eigenvalue.imag, residual, offset, readout,
                ])
                printable.append(
                    f"sign={sign:+d}: |lambda|={abs(eigenvalue):.6f}, "
                    f"offset/T={offset:+.6f}, residual={residual:.2e}, "
                    f"edge={readout:.3f}"
                )
            history.append(report_row)
            print(
                f"  m={used_dimension:3d}, elapsed="
                f"{time.perf_counter() - started:.1f}s | " + " | ".join(printable)
            )

    history_array = np.asarray(history, dtype=float)
    selected_records = []
    for sign, start_column in [(1, 2), (-1, 7)]:
        branch_rows = np.where(np.isfinite(history_array[:, start_column + 2]))[0]
        if len(branch_rows) == 0:
            continue
        best_row_index = branch_rows[
            np.argmin(history_array[branch_rows, start_column + 2])
        ]
        best_row = history_array[best_row_index]
        eigenvalue = best_row[start_column] + 1j * best_row[start_column + 1]
        residual = float(best_row[start_column + 2])
        offset = float(best_row[start_column + 3])
        readout = float(best_row[start_column + 4])
        best_dimension = float(best_row[1])
        neighbor_indices = [
            index for index in [best_row_index - 1, best_row_index + 1]
            if 0 <= index < len(history_array)
            and np.isfinite(history_array[index, start_column])
        ]
        neighbor_drift = (
            float(max(
                abs(
                    eigenvalue
                    - (
                        history_array[index, start_column]
                        + 1j * history_array[index, start_column + 1]
                    )
                )
                for index in neighbor_indices
            ))
            if neighbor_indices else np.nan
        )
        modulus = float(abs(eigenvalue))
        lifetime_periods = (
            float(-1.0 / np.log(modulus)) if 0.0 < modulus < 1.0 else np.nan
        )
        selected_records.append([
            float(parameters.g), float(sign),
            float(eigenvalue.real), float(eigenvalue.imag), modulus,
            offset, lifetime_periods, residual, readout,
            best_dimension, neighbor_drift,
            float(
                residual <= residual_tol
                and modulus <= 1.0 + residual_tol
                and (np.isnan(neighbor_drift) or neighbor_drift <= 2e-3)
            ),
        ])
    return {
        "selected_records": np.asarray(selected_records, dtype=float),
        "history": history_array,
        "used_dimension": used_dimension,
        "runtime_seconds": time.perf_counter() - started,
    }


def validate_pi_arnoldi_against_exact_n4(krylov_dim=200, residual_tol=1e-3):
    parameters = replace(
        p, N=4, g=0.08, tls_site=0,
        omega_d=p.Omega / 2.0, periods=8,
    )
    result = edge_seeded_pi_arnoldi(
        parameters, krylov_dim=krylov_dim,
        residual_tol=residual_tol, report_every=50,
    )
    if not RUN_EXACT_N4_CHANNEL:
        print("Exact N=4 comparison unavailable because its flag is false.")
        return result
    spectrum = exact_n4_spectra[0.08]
    comparisons = []
    for row in result["selected_records"]:
        sign = int(row[1])
        tracked = next(
            item for item in exact_n4_tracked[sign]
            if np.isclose(item["g"], 0.08)
        )
        exact_value = spectrum["eigenvalues"][tracked["index"]]
        arnoldi_value = row[2] + 1j * row[3]
        comparisons.append([
            sign, exact_value.real, exact_value.imag,
            arnoldi_value.real, arnoldi_value.imag,
            abs(arnoldi_value - exact_value), row[7],
        ])
    print("N=4 [sign, exact Re, exact Im, Arnoldi Re, Arnoldi Im, |difference|, residual]")
    print(np.asarray(comparisons))
    return result


def run_long_n6_matrix_free_channel(
    krylov_dim=200, residual_tol=1e-3, report_every=20,
):
    """Checkpoint after each coupling; an interrupted coupling is safely retried."""
    checkpoint_directory = Path(CHECKPOINT_DIRECTORY).expanduser().resolve()
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    path = checkpoint_directory / "floquet_tls_N6_edge_seeded_pi_arnoldi_v2_checkpoint.npz"
    coupling_values = np.asarray([0.06, 0.08, 0.10, 0.12])
    algorithm = "edge_seeded_unrestarted_pi_arnoldi_v2_best_residual"
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            records = saved["records"].tolist()
            history = saved["history"].tolist()
            completed_g = saved["completed_g"].tolist()
            stored_algorithm = str(saved["metadata_algorithm"])
            stored_dimension = int(saved["metadata_krylov_dim"])
            stored_tolerance = float(saved["metadata_residual_tol"])
        if (
            stored_algorithm != algorithm
            or stored_dimension != krylov_dim
            or not np.isclose(stored_tolerance, residual_tol)
        ):
            raise ValueError("Arnoldi checkpoint settings mismatch")
        print("resuming channel checkpoint:", path, "completed g =", completed_g)
    else:
        records, history, completed_g = [], [], []
        print("new channel checkpoint:", path)

    for coupling in coupling_values:
        if any(np.isclose(coupling, completed) for completed in completed_g):
            continue
        parameters = replace(
            p, N=6, g=float(coupling), tls_site=0,
            omega_d=p.Omega / 2.0, periods=8,
        )
        result = edge_seeded_pi_arnoldi(
            parameters, krylov_dim=krylov_dim,
            residual_tol=residual_tol, report_every=report_every,
        )
        records.extend(result["selected_records"].tolist())
        history.extend(result["history"].tolist())
        completed_g.append(float(coupling))
        atomic_savez(
            path,
            records=np.asarray(records, dtype=float),
            history=np.asarray(history, dtype=float),
            completed_g=np.asarray(completed_g, dtype=float),
            record_columns=np.asarray([
                "g", "branch_sign", "lambda_real", "lambda_imag", "modulus",
                "phase_offset_over_T", "lifetime_periods", "ritz_residual",
                "edge_readout", "krylov_dim", "neighbor_eigenvalue_drift",
                "accepted",
            ]),
            history_columns=np.asarray([
                "g", "krylov_dim",
                "positive_real", "positive_imag", "positive_residual",
                "positive_offset_over_T", "positive_edge_readout",
                "negative_real", "negative_imag", "negative_residual",
                "negative_offset_over_T", "negative_edge_readout",
            ]),
            metadata_algorithm=np.asarray(algorithm),
            metadata_krylov_dim=np.asarray(krylov_dim),
            metadata_residual_tol=np.asarray(residual_tol),
            metadata_report_every=np.asarray(report_every),
        )
        selected_array = result["selected_records"]
        accepted = (
            selected_array[:, -1].astype(bool)
            if selected_array.ndim == 2 and len(selected_array) else np.asarray([], dtype=bool)
        )
        print(
            f"saved g={coupling:.2f} after {result['runtime_seconds']:.1f}s; "
            f"accepted branches={int(accepted.sum())}/{len(accepted)}"
        )
    return np.asarray(records, dtype=float)


if RUN_EXACT_N4_CHANNEL:
    print("Exact N=4 dense channel grid completed in the benchmark cell above.")
else:
    print("Exact N=4 dense channel grid skipped.")

if RUN_N4_ARNOLDI_VALIDATION:
    n4_arnoldi_validation = validate_pi_arnoldi_against_exact_n4()
else:
    print("N=4 targeted-Arnoldi validation skipped.")

if RUN_LONG_N6_POSITION_GRID:
    long_position_data = run_long_n6_position_grid()
else:
    print("Long N=6 position-frequency grid disabled for local execution.")

if RUN_LONG_N6_MATRIX_FREE_CHANNEL:
    long_channel_records = run_long_n6_matrix_free_channel()
else:
    print("Long N=6 targeted-Arnoldi channel disabled for local execution.")


## 8. Acceptance rules

### Dressed spatial selectivity

A topology-linked position claim requires:

1. one fixed (0-π) pair or a clearly defined spectral projector;
2. a stated Floquet sideband (m);
3. a two-edge spatial form on a finite open chain;
4. stability against time discretization and pair-selection tolerance;
5. consistency with, but not automatic equality to, the independently obtained Majorana localization length.

### Channel validation

A channel mode is evidence for metastable period doubling only when its phase is close to π, its modulus and lifetime are converged, and it has non-negligible initial/readout visibility. A dense (N=3) result or a single-doublet reduced model is a benchmark, not a production-size proof.

### Open position response

The spatial test passes only if the **frequency-resolved** response and a robust spectral statistic are boundary enhanced. Failure of a fixed-frequency amplitude to decay monotonically is not by itself a contradiction, because hybridization moves spectral peaks and a finite chain contains two boundaries.
